### В этом ноутбуке основной анализ данных (воспроизведение таблиц из статьи)

(Подготовку данных можно посмотреть в другом ноутбуке указаном в README)


Сначала импорты

In [91]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install statsmodels
%pip install scipy
%pip install pyarrow
%pip install fastparquet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [92]:
import pandas as pd
import os
PARQUET_FILES_DIR = "data_for_analysis"

us_full_time = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "us_processed_full_time.parquet"), engine="fastparquet")
us_sub1 = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "us_processed_sub1.parquet"), engine="fastparquet")
us_sub2 = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "us_processed_sub2.parquet"), engine="fastparquet")

uk_full_time = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "uk_processed_full_time.parquet"), engine="fastparquet")
uk_sub1 = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "uk_processed_sub1.parquet"), engine="fastparquet")
uk_sub2 = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "uk_processed_sub2.parquet"), engine="fastparquet")

bd_full_time = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "bd_processed_full_time.parquet"), engine="fastparquet")
bd_sub1 = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "bd_processed_sub1.parquet"), engine="fastparquet")
bd_sub2 = pd.read_parquet(os.path.join(PARQUET_FILES_DIR, "bd_processed_sub2.parquet"), engine="fastparquet")


Теперь фунция для подсчета статистик и параметров из первой таблицы статьи (table 1)

In [93]:
from scipy.stats import skew, kurtosis, jarque_bera
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

def calculate_table1_stats(series):
    s = series.dropna()
    
    stats = {
        "Mean": s.mean(),
        "Median": s.median(),
        "Std. Dev.": s.std(),
        "Skewness": skew(s),
        "Kurtosis": kurtosis(s, fisher=False) 
    }

    jb_stat, jb_pvalue = jarque_bera(s)
    stats["JB"] = jb_stat

    lb_test = acorr_ljungbox(s, lags=[1, 5], return_df=True)
    stats["LB(1)"] = lb_test['lb_stat'].iloc[0]
    stats["LB(5)"] = lb_test['lb_stat'].iloc[1]

    adf_result = adfuller(s, maxlag=10, autolag=None, result_object=False)
    stats["ADF(10)"] = adf_result[0]

    return pd.Series(stats)


Таблица 1 для США

In [94]:
TIME_PERIODS = {
    "full_time":["1992-01-03", "2000-04-20"],
    "sub1":["1992-01-03", "1997-05-30"],
    "sub2":["1997-06-02", "2000-04-20"]
    }

table_1_panel_a_us = pd.DataFrame({
    "ERUS": calculate_table1_stats(us_full_time["ER_US"]),
    "YUS": calculate_table1_stats(us_full_time["Y_US"]),
    "DLDJIA": calculate_table1_stats(us_full_time["s_US"]),
})
table_1_panel_b_us = pd.DataFrame({
    "ERUS": calculate_table1_stats(us_sub1["ER_US"]),
    "YUS": calculate_table1_stats(us_sub1["Y_US"]),
    "DLDJIA": calculate_table1_stats(us_sub1["s_US"]),
})
table_1_panel_c_us = pd.DataFrame({
    "ERUS": calculate_table1_stats(us_sub2["ER_US"]),
    "YUS": calculate_table1_stats(us_sub2["Y_US"]),
    "DLDJIA": calculate_table1_stats(us_sub2["s_US"]),
})

print(f"panel a:{TIME_PERIODS["full_time"]}\n")
print(table_1_panel_a_us.round(4))
print("--------------")
print(f"panel b:{TIME_PERIODS["sub1"]}\n")
print(table_1_panel_b_us.round(4))
print("--------------")
print(f"panel c:{TIME_PERIODS["sub2"]}\n")
print(table_1_panel_c_us.round(4))

panel a:['1992-01-03', '2000-04-20']

                ERUS       YUS     DLDJIA
Mean          0.0420    0.0029     0.0592
Median        0.0480    0.0000     0.0659
Std. Dev.     0.9181    0.4435     0.9181
Skewness     -0.5329   -0.4307    -0.5355
Kurtosis      9.1614    5.6174     9.1667
JB         3383.6633  657.0940  3390.2493
LB(1)         0.4997    8.2107     0.4930
LB(5)         8.8029   23.5971     8.8425
ADF(10)     -13.9927  -13.8173   -14.0028
--------------
panel b:['1992-01-03', '1997-05-30']

               ERUS       YUS    DLDJIA
Mean         0.0438    0.0006    0.0619
Median       0.0494    0.0000    0.0670
Std. Dev.    0.6847    0.4526    0.6846
Skewness    -0.2629   -0.5583   -0.2624
Kurtosis     4.5521    6.3029    4.5533
JB         151.3963  685.3084  151.5447
LB(1)        2.8544    3.9176    2.8351
LB(5)       13.7584   14.5592   13.7921
ADF(10)    -11.1977  -11.4282  -11.2078
--------------
panel c:['1997-06-02', '2000-04-20']

               ERUS      YUS    DLDJ

То же самое для Великобритании и Германии

In [95]:
table_1_panel_a_uk = pd.DataFrame({
    "ERUK": calculate_table1_stats(uk_full_time["ER_UK"]),
    "YUK": calculate_table1_stats(uk_full_time["Y_UK"]),
    "DLFTSE": calculate_table1_stats(uk_full_time["s_UK"]),
})
table_1_panel_b_uk = pd.DataFrame({
    "ERUK": calculate_table1_stats(uk_sub1["ER_UK"]),
    "YUK": calculate_table1_stats(uk_sub1["Y_UK"]),
    "DLFTSE": calculate_table1_stats(uk_sub1["s_UK"]),
})
table_1_panel_c_uk = pd.DataFrame({
    "ERUK": calculate_table1_stats(uk_sub2["ER_UK"]),
    "YUK": calculate_table1_stats(uk_sub2["Y_UK"]),
    "DLFTSE": calculate_table1_stats(uk_sub2["s_UK"]),
})

print(f"panel a:{TIME_PERIODS["full_time"]}\n")
print(table_1_panel_a_uk.round(4))
print("--------------")
print(f"panel b:{TIME_PERIODS["sub1"]}\n")
print(table_1_panel_b_uk.round(4))
print("--------------")
print(f"panel c:{TIME_PERIODS["sub2"]}\n")
print(table_1_panel_c_uk.round(4))

panel a:['1992-01-03', '2000-04-20']

               ERUK        YUK    DLFTSE
Mean         0.0241     0.0145    0.0437
Median       0.0295     0.0271    0.0484
Std. Dev.    0.9459     0.4794    0.9459
Skewness     0.0123    -0.1065    0.0068
Kurtosis     5.1148     6.8552    5.1190
JB         391.2158  1303.7991  392.7359
LB(1)       14.7495     0.3512   14.7334
LB(5)       27.8077     2.4863   27.8226
ADF(10)    -13.8993   -13.9417  -13.9041
--------------
panel b:['1992-01-03', '1997-05-30']

               ERUK       YUK    DLFTSE
Mean         0.0231    0.0116    0.0452
Median       0.0193    0.0301    0.0431
Std. Dev.    0.7441    0.5093    0.7440
Skewness     0.2449   -0.0173    0.2472
Kurtosis     6.5719    6.9952    6.5786
JB         740.3552  909.2017  743.3546
LB(1)        3.6575    0.1161    3.6372
LB(5)        4.5246    2.6352    4.4894
ADF(10)    -11.5843  -11.5826  -11.5928
--------------
panel c:['1997-06-02', '2000-04-20']

              ERUK       YUK   DLFTSE
Mean    

In [96]:
table_1_panel_a_bd = pd.DataFrame({
    "ERBD": calculate_table1_stats(bd_full_time["ER_BD"]),
    "YBD": calculate_table1_stats(bd_full_time["Y_BD"]),
    "DLDAX": calculate_table1_stats(bd_full_time["s_BD"]),
})
table_1_panel_b_bd = pd.DataFrame({
    "ERBD": calculate_table1_stats(bd_sub1["ER_BD"]),
    "YBD": calculate_table1_stats(bd_sub1["Y_BD"]),
    "DLDAX": calculate_table1_stats(bd_sub1["s_BD"]),
})
table_1_panel_c_bd = pd.DataFrame({
    "ERBD": calculate_table1_stats(bd_sub2["ER_BD"]),
    "YBD": calculate_table1_stats(bd_sub2["Y_BD"]),
    "DLDAX": calculate_table1_stats(bd_sub2["s_BD"]),
})

print(f"panel a:{TIME_PERIODS["full_time"]}\n")
print(table_1_panel_a_bd.round(4))
print("--------------")
print(f"panel b:{TIME_PERIODS["sub1"]}\n")
print(table_1_panel_b_bd.round(4))
print("--------------")
print(f"panel c:{TIME_PERIODS["sub2"]}\n")
print(table_1_panel_c_bd.round(4))

panel a:['1992-01-03', '2000-04-20']

               ERBD       YBD     DLDAX
Mean         0.0558    0.0087    0.0724
Median       0.0857    0.0221    0.1033
Std. Dev.    1.2286    0.3588    1.2285
Skewness    -0.3557   -0.4679   -0.3601
Kurtosis     6.1632    5.5953    6.1696
JB         905.3314  655.5414  909.9055
LB(1)        2.1382    0.1866    2.1226
LB(5)        8.4179    1.5810    8.4156
ADF(10)    -13.6635  -13.8849  -13.6754
--------------
panel b:['1992-01-03', '1997-05-30']

               ERBD       YBD     DLDAX
Mean         0.0411    0.0104    0.0595
Median       0.0673    0.0227    0.0881
Std. Dev.    0.9122    0.3578    0.9120
Skewness    -0.3014   -0.5268   -0.3020
Kurtosis     4.7465    6.0755    4.7500
JB         191.0160  591.3961  191.7861
LB(1)        0.0225    0.5038    0.0191
LB(5)        6.1356    5.4133    6.1116
ADF(10)    -11.4099  -11.3938  -11.4342
--------------
panel c:['1997-06-02', '2000-04-20']

              ERBD       YBD    DLDAX
Mean        0.0831

Дальше итоговая таблица 1 из статьи

In [97]:
table_1_panel_a_full = pd.concat([table_1_panel_a_us, table_1_panel_a_uk, table_1_panel_a_bd], axis=1)
print(f"TABLE 1\n{TIME_PERIODS["full_time"]}")
print(table_1_panel_a_full.round(4))
print("\n-------------------\n")
table_1_panel_b_full = pd.concat([table_1_panel_b_us, table_1_panel_b_uk, table_1_panel_b_bd], axis=1)
print(f"{TIME_PERIODS["sub1"]}")
print(table_1_panel_b_full.round(4))
print("\n-------------------\n")
table_1_panel_c_full = pd.concat([table_1_panel_c_us, table_1_panel_c_uk, table_1_panel_c_bd], axis=1)
print(f"{TIME_PERIODS["sub2"]}")
print(table_1_panel_c_full.round(4))

TABLE 1
['1992-01-03', '2000-04-20']
                ERUS       YUS     DLDJIA      ERUK        YUK    DLFTSE  \
Mean          0.0420    0.0029     0.0592    0.0241     0.0145    0.0437   
Median        0.0480    0.0000     0.0659    0.0295     0.0271    0.0484   
Std. Dev.     0.9181    0.4435     0.9181    0.9459     0.4794    0.9459   
Skewness     -0.5329   -0.4307    -0.5355    0.0123    -0.1065    0.0068   
Kurtosis      9.1614    5.6174     9.1667    5.1148     6.8552    5.1190   
JB         3383.6633  657.0940  3390.2493  391.2158  1303.7991  392.7359   
LB(1)         0.4997    8.2107     0.4930   14.7495     0.3512   14.7334   
LB(5)         8.8029   23.5971     8.8425   27.8077     2.4863   27.8226   
ADF(10)     -13.9927  -13.8173   -14.0028  -13.8993   -13.9417  -13.9041   

               ERBD       YBD     DLDAX  
Mean         0.0558    0.0087    0.0724  
Median       0.0857    0.0221    0.1033  
Std. Dev.    1.2286    0.3588    1.2285  
Skewness    -0.3557   -0.4679   -0

Теперь таблица 2 (Granger causality test between the stock market excess return and the expected rate of change of long-term
government bond prices)

In [98]:
us_full_time.head()

,Date,Price,BondYield,s_US,i_US,delta_i_US,Y_US,ER_US
0,1992-01-03,3201.5000,6.85,0.913108,0.018767,0.07,-0.525,0.894341
1,1992-01-06,3200.1001,6.82,-0.043736,0.018685,-0.03,0.225,-0.062421
2,1992-01-07,3204.8000,6.76,0.146760,0.018521,-0.06,0.450,0.128239
3,1992-01-08,3203.8999,6.77,-0.028090,0.018548,0.01,-0.075,-0.046638
4,1992-01-09,3209.5000,6.79,0.174638,0.018603,0.02,-0.150,0.156035


In [99]:
from statsmodels.tsa.stattools import grangercausalitytests


def granger_pair(df, cause_col, effect_col, lags=(2, 5, 10)):
    sub = df[[effect_col, cause_col]].dropna()
    results = []
    for lag in lags:
        res = grangercausalitytests(sub, maxlag=[lag])
        f_stat = res[lag][0]["ssr_ftest"][0]
        p_val = res[lag][0]["ssr_ftest"][1]
        results.append((lag, f_stat, p_val))
    return results


def run_country_table(df, country_suffix, lags=(2, 5, 10)):
    er_col = f"ER_{country_suffix}"
    y_col = f"Y_{country_suffix}"

    er_to_y = granger_pair(df, cause_col=er_col, effect_col=y_col, lags=lags)
    y_to_er = granger_pair(df, cause_col=y_col, effect_col=er_col, lags=lags)

    print(f"\n=== {country_suffix} ===")
    print(f"{'Lags':<6}{'ER->Y (F, p)':<25}{'Y->ER (F, p)':<25}")
    for (lag1, f1, p1), (lag2, f2, p2) in zip(er_to_y, y_to_er):
        assert lag1 == lag2
        star1 = "*" if p1 < 0.05 else ("**" if p1 < 0.10 else "")
        star2 = "*" if p2 < 0.05 else ("**" if p2 < 0.10 else "")
        print(f"{lag1:<6}{f1:.2f} [{p1:.2f}]{star1:<10}{f2:.2f} [{p2:.2f}]{star2:<10}")

    return {
        f"{er_col}->{y_col}": er_to_y,
        f"{y_col}->{er_col}": y_to_er,
    }


datasets = {
    "US": us_full_time,
    "UK": uk_full_time,
    "BD": bd_full_time,
}

print(f"TABLE 2\n{TIME_PERIODS['full_time']}")
all_results = {}
for suffix, df in datasets.items():
    all_results[suffix] = run_country_table(df, suffix, lags=(2, 5, 10))

TABLE 2
['1992-01-03', '2000-04-20']

=== US ===
Lags  ER->Y (F, p)             Y->ER (F, p)             
2     0.78 [0.46]          3.26 [0.04]*         
5     1.62 [0.15]          2.14 [0.06]**        
10    1.12 [0.34]          1.40 [0.18]          

=== UK ===
Lags  ER->Y (F, p)             Y->ER (F, p)             
2     0.49 [0.61]          2.90 [0.06]**        
5     1.27 [0.27]          3.26 [0.01]*         
10    1.20 [0.29]          1.96 [0.03]*         

=== BD ===
Lags  ER->Y (F, p)             Y->ER (F, p)             
2     1.28 [0.28]          10.50 [0.00]*         
5     0.76 [0.58]          4.60 [0.00]*         
10    1.00 [0.44]          3.06 [0.00]*         


В целом результаты сходятся, есть небольшие расхождения (например с США), но выводам статьи они не противоречат. Эти расхождения обусловлены тем, что цены фьючерсов не получилось найти и пришлось идти на упрощение через дюрацию.

Далее найдем коэффициенты OSL 

In [118]:
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_arch
 
 
def run_ols_equation7(df, country_prefix):
    er_col = f"ER_{country_prefix}"
    y_col = f"Y_{country_prefix}"
 
    sub = df[[er_col, y_col]].dropna()
 
    X = sm.add_constant(sub[y_col])
    y = sub[er_col]
    model = sm.OLS(y, X).fit(cov_type="HC0")
    a_hat, b_hat = model.params["const"], model.params[y_col]
    a_se, b_se = model.bse["const"], model.bse[y_col]
    r2 = model.rsquared
    dw = durbin_watson(model.resid)
    arch_stat, arch_pval, _, _ = het_arch(model.resid, nlags=12, result_object=False)
 
    return {
        "country": country_prefix,
        "a": a_hat,
        "a_se": a_se,
        "a_pval": model.pvalues["const"],
        "b": b_hat,
        "b_se": b_se,
        "b_pval": model.pvalues[y_col],
        "R2": r2,
        "DW": dw,
        "Arch12_stat": arch_stat,
        "Arch12_pval": arch_pval,
        "residuals": model.resid,
        "model": model,
    }

In [119]:
def print_table3(results):
    print(f"{'':<15}{'US':<20}{'UK':<20}{'BD':<20}")
 
    def fmt_coef(r, key, se_key, pval_key):
        star = "*" if r[pval_key] < 0.05 else ""
        return f"{r[key]:.3f} ({r[se_key]:.2f}){star}"
 
    rows = [
        ("a (s.e.)", lambda r: fmt_coef(r, "a", "a_se", "a_pval")),
        ("b (s.e.)", lambda r: fmt_coef(r, "b", "b_se", "b_pval")),
        ("R2", lambda r: f"{r['R2']:.3f}"),
        ("DW", lambda r: f"{r['DW']:.2f}"),
        ("Arch(12) [prob]", lambda r: f"{r['Arch12_stat']:.2f} [{r['Arch12_pval']:.2f}]"),
    ]
 
    for label, fn in rows:
        line = f"{label:<15}"
        for country in ["US", "UK", "BD"]:
            line += f"{fn(results[country]):<20}"
        print(line)

In [120]:
datasets = {
        "US": us_full_time,
        "UK": uk_full_time,
        "BD": bd_full_time,
        }
results = {}
exuberance = {}

for suffix, df in datasets.items():
    res = run_ols_equation7(df, suffix)
    results[suffix] = res
    exuberance[f"EX_{suffix}"] = res["residuals"]
    print(f"\n{suffix}: a={res['a']:.4f} b={res['b']:.4f} R2={res['R2']:.4f} "
            f"DW={res['DW']:.2f} Arch(12)={res['Arch12_stat']:.2f} "
            f"[p={res['Arch12_pval']:.3f}]")

print("\nTABLE 3")
print_table3(results)


US: a=0.0409 b=0.3749 R2=0.0328 DW=1.96 Arch(12)=269.72 [p=0.000]

UK: a=0.0157 b=0.5750 R2=0.0849 DW=1.81 Arch(12)=362.04 [p=0.000]

BD: a=0.0503 b=0.6310 R2=0.0340 DW=1.98 Arch(12)=320.45 [p=0.000]

TABLE 3
               US                  UK                  BD                  
a (s.e.)       0.041 (0.02)*       0.016 (0.02)        0.050 (0.03)        
b (s.e.)       0.375 (0.05)*       0.575 (0.06)*       0.631 (0.09)*       
R2             0.033               0.085               0.034               
DW             1.96                1.81                1.98                
Arch(12) [prob]269.72 [0.00]       362.04 [0.00]       320.45 [0.00]       


Теперь можем получить EX как остатки линейной модели

In [121]:
us_full_time.loc[results["US"]["residuals"].index, "EX_US"] = results["US"]["residuals"]
uk_full_time.loc[results["UK"]["residuals"].index, "EX_UK"] = results["UK"]["residuals"]
bd_full_time.loc[results["BD"]["residuals"].index, "EX_BD"] = results["BD"]["residuals"]

In [124]:
from scipy import stats as sps
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
 
 
PANELS = {
    "a (full sample)": (TIME_PERIODS["full_time"][0], TIME_PERIODS["full_time"][1]),
    "b (pre-crisis)": (TIME_PERIODS["sub1"][0], TIME_PERIODS["sub1"][1]),
    "c (post-crisis)": (TIME_PERIODS["sub2"][0], TIME_PERIODS["sub2"][1]),
}
 
 
def descriptive_stats(series, adf_lags=10):
    s = series.dropna()
    mean = s.mean()
    median = s.median()
    std = s.std(ddof=1)
    skew = sps.skew(s)
    kurt = sps.kurtosis(s, fisher=False)
 
    jb_stat, jb_pval, _, _ = jarque_bera(s)
 
    lb = acorr_ljungbox(s, lags=[1, 5], return_df=True)
    lb1_stat, lb1_pval = lb.loc[1, "lb_stat"], lb.loc[1, "lb_pvalue"]
    lb5_stat, lb5_pval = lb.loc[5, "lb_stat"], lb.loc[5, "lb_pvalue"]
 
    adf_res = adfuller(s, maxlag=adf_lags, autolag=None, regression="c", result_object=False)
    adf_stat, adf_pval = adf_res[0], adf_res[1]
 
    return {
        "Mean": mean,
        "Median": median,
        "Std.Dev": std,
        "Skewness": skew,
        "Kurtosis": kurt,
        "JB_stat": jb_stat,
        "JB_pval": jb_pval,
        "LB1_stat": lb1_stat,
        "LB1_sig": lb1_pval < 0.05,
        "LB5_stat": lb5_stat,
        "LB5_sig": lb5_pval < 0.05,
        "ADF_stat": adf_stat,
        "ADF_sig": adf_pval < 0.05,
        "n_obs": len(s),
    }
 
 
def build_table4(df_dict, panels=PANELS, date_col="Date"):
    results = {}
    for panel_name, (start, end) in panels.items():
        results[panel_name] = {}
        for country, df in df_dict.items():
            mask = (df[date_col] >= start) & (df[date_col] <= end)
            sub = df.loc[mask, f"EX_{country}"]
            results[panel_name][country] = descriptive_stats(sub)
    return results
 
 
def print_table4(results):
    for panel_name, countries in results.items():
        print(f"\n=== Panel {panel_name} ===")
        header = f"{'':<12}" + "".join(f"{c:<16}" for c in countries)
        print(header)
 
        rows = ["Mean", "Median", "Std.Dev", "Skewness", "Kurtosis"]
        for r in rows:
            line = f"{r:<12}"
            for c, stats_dict in countries.items():
                line += f"{stats_dict[r]:<16.3f}"
            print(line)
 
        line = f"{'JB':<12}"
        for c, s in countries.items():
            star = "*" if s["JB_pval"] < 0.05 else ""
            line += f"{s['JB_stat']:.1f}{star:<12}"
        print(line)
 
        line = f"{'LB(1)':<12}"
        for c, s in countries.items():
            star = "**" if s["LB1_sig"] else ""
            line += f"{s['LB1_stat']:.2f}{star:<12}"
        print(line)
 
        line = f"{'LB(5)':<12}"
        for c, s in countries.items():
            star = "**" if s["LB5_sig"] else ""
            line += f"{s['LB5_stat']:.2f}{star:<12}"
        print(line)
 
        line = f"{'ADF(10)':<12}"
        for c, s in countries.items():
            star = "*" if s["ADF_sig"] else ""
            line += f"{s['ADF_stat']:.2f}{star:<12}"
        print(line)
 
        line = f"{'N obs':<12}"
        for c, s in countries.items():
            line += f"{s['n_obs']:<16}"
        print(line)
 

 
df_dict = {
    "US": us_full_time,
    "UK": uk_full_time,
    "BD": bd_full_time,
}

results = build_table4(df_dict)
print_table4(results)


=== Panel a (full sample) ===
            US              UK              BD              
Mean        0.000           0.000           0.000           
Median      0.001           0.005           0.019           
Std.Dev     0.903           0.905           1.208           
Skewness    -0.665          -0.021          -0.430          
Kurtosis    10.645          6.044           6.816           
JB          5210.9*           810.5*           1318.1*           
LB(1)       0.60            18.64**          0.26            
LB(5)       7.60            47.42**          6.01            
ADF(10)     -13.86*           -14.22*           -13.50*           
N obs       2077            2099            2067            

=== Panel b (pre-crisis) ===
            US              UK              BD              
Mean        0.003           0.001           -0.016          
Median      -0.002          0.004           0.005           
Std.Dev     0.633           0.637           0.871           
Skewness   

Результаты сходятся со статьей

Переходим к таблице 5

In [130]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.stats.diagnostic import het_arch
 
 
BREAK_DATE = "1997-05-30"  
 
 
def build_dummy(dates, break_date=BREAK_DATE):
    dates = pd.to_datetime(dates)
    return (dates > pd.Timestamp(break_date)).astype(float).values
 
 
def tgarch_recursion(params, u, D):
    a0, a1, a2, gamma, phi = params
    T = len(u)
    h = np.zeros(T)
    h[0] = np.var(u)  
 
    for t in range(T - 1):
        S_t = 1.0 if u[t] < 0 else 0.0
        h_next = a0 + a1 * u[t] ** 2 + a2 * h[t] + gamma * S_t * u[t] ** 2 + phi * D[t + 1]
        h[t + 1] = max(h_next, 1e-10) 
 
    return h
 
 
def neg_loglik(params, u, D):
    a0, a1, a2, gamma, phi = params
    if a0 <= 0 or a1 < 0 or a2 < 0 or a1 + a2 + gamma / 2 >= 1:
        return 1e10
    h = tgarch_recursion(params, u, D)
    if np.any(h <= 0) or np.any(~np.isfinite(h)):
        return 1e10
    ll = -0.5 * np.log(2 * np.pi) - 0.5 * np.log(h) - 0.5 * (u ** 2) / h
    return -np.sum(ll)
 
 
def numerical_hessian(f, x0, args, eps=1e-4):
    n = len(x0)
    H = np.zeros((n, n))
    f0 = f(x0, *args)
    for i in range(n):
        for j in range(n):
            xi = x0.copy(); xi[i] += eps
            xj = x0.copy(); xj[j] += eps
            xij = x0.copy(); xij[i] += eps; xij[j] += eps
            f_i = f(xi, *args)
            f_j = f(xj, *args)
            f_ij = f(xij, *args)
            H[i, j] = (f_ij - f_i - f_j + f0) / (eps ** 2)
    return H
 
 
def numerical_gradient_per_obs(params, u, D, eps=1e-5):
    a0, a1, a2, gamma, phi = params
    n = len(params)
    T = len(u)
    grad = np.zeros((T, n))
 
    for k in range(n):
        p_plus = params.copy(); p_plus[k] += eps
        p_minus = params.copy(); p_minus[k] -= eps
        h_plus = tgarch_recursion(p_plus, u, D)
        h_minus = tgarch_recursion(p_minus, u, D)
        ll_plus = -0.5 * np.log(2 * np.pi) - 0.5 * np.log(h_plus) - 0.5 * (u ** 2) / h_plus
        ll_minus = -0.5 * np.log(2 * np.pi) - 0.5 * np.log(h_minus) - 0.5 * (u ** 2) / h_minus
        grad[:, k] = (ll_plus - ll_minus) / (2 * eps)
    return grad
 
 
def fit_tgarch(u, D, x0=None):
    if x0 is None:
        var_u = np.var(u)
        x0 = np.array([0.05 * var_u, 0.05, 0.85, 0.05, 0.0])
 
    res = minimize(
        neg_loglik, x0, args=(u, D),
        method="L-BFGS-B",
        bounds=[(1e-8, None), (0, 1), (0, 1), (-1, 1), (-2, 2)],
        options={"maxiter": 2000, "ftol": 1e-12},
    )
    params = res.x
    H_neg = numerical_hessian(neg_loglik, params, (u, D))  
    H = -H_neg
    grad_i = numerical_gradient_per_obs(params, u, D)
    OPG = grad_i.T @ grad_i
 
    H_inv = np.linalg.inv(H_neg)
    cov = H_inv @ OPG @ H_inv
    se = np.sqrt(np.diag(cov))
 
    h_final = tgarch_recursion(params, u, D)
 
    return {
        "params": params,
        "se": se,
        "h": h_final,
        "loglik": -res.fun,
        "success": res.success,
    }
 
 
def print_table5_row(country, fit_result, u, D):
    names = ["a0", "a1", "a2", "gamma", "phi"]
    params, se = fit_result["params"], fit_result["se"]
 
    print(f"\n=== {country} ===")
    for name, p, s in zip(names, params, se):
        t_stat = p / s if s > 0 else np.nan
        star = "*" if abs(t_stat) > 1.96 else ""
        print(f"  {name:<8} {p: .4f}  (s.e. {s:.4f})  t={t_stat:6.2f} {star}")
 
    std_resid = u / np.sqrt(fit_result["h"])
    arch1_stat, arch1_p, _, _ = het_arch(std_resid, nlags=1, result_object=False)
    arch12_stat, arch12_p, _, _ = het_arch(std_resid, nlags=12, result_object=False)
 
    print(f"  Arch(1)  [prob] {arch1_stat:.2f} [{arch1_p:.2f}]")
    print(f"  Arch(12) [prob] {arch12_stat:.2f} [{arch12_p:.2f}]")
 
    return {
        "params": dict(zip(names, params)),
        "se": dict(zip(names, se)),
        "arch1": (arch1_stat, arch1_p),
        "arch12": (arch12_stat, arch12_p),
    }
 
datasets = {
    "US": us_full_time,
    "UK": uk_full_time,
    "BD": bd_full_time,
}

table5_results = {}

for country, df in datasets.items():
    df_sorted = df.sort_values("Date").dropna(subset=[f"EX_{country}"])
    u = df_sorted[f"EX_{country}"].values
    D = build_dummy(df_sorted["Date"])

    fit = fit_tgarch(u, D)
    table5_results[country] = print_table5_row(country, fit, u, D)


=== US ===
  a0        0.0286  (s.e. 0.0112)  t=  2.56 *
  a1        0.0196  (s.e. 0.0147)  t=  1.34 
  a2        0.8441  (s.e. 0.0441)  t= 19.16 *
  gamma     0.1355  (s.e. 0.0515)  t=  2.63 *
  phi       0.0674  (s.e. 0.0269)  t=  2.50 *
  Arch(1)  [prob] 0.21 [0.65]
  Arch(12) [prob] 7.74 [0.81]

=== UK ===
  a0        0.0072  (s.e. 0.0044)  t=  1.64 
  a1        0.0152  (s.e. 0.0136)  t=  1.12 
  a2        0.9395  (s.e. 0.0225)  t= 41.68 *
  gamma     0.0541  (s.e. 0.0143)  t=  3.79 *
  phi       0.0219  (s.e. 0.0127)  t=  1.73 
  Arch(1)  [prob] 1.05 [0.31]
  Arch(12) [prob] 7.63 [0.81]

=== BD ===
  a0        0.0315  (s.e. 0.0107)  t=  2.94 *
  a1        0.0394  (s.e. 0.0174)  t=  2.26 *
  a2        0.8805  (s.e. 0.0216)  t= 40.68 *
  gamma     0.0818  (s.e. 0.0322)  t=  2.54 *
  phi       0.0683  (s.e. 0.0230)  t=  2.97 *
  Arch(1)  [prob] 0.03 [0.86]
  Arch(12) [prob] 7.93 [0.79]
